# 동물·사람 탐지 YOLO 학습 (Colab)

초안입니다. 2026-11-12 모델을 만든 실제 노트북이 있으면 이 파일을 그것으로 바꿔 주세요.

순서: Drive 연결 → 설치 → 학습 → 평가 → NCNN 변환. 런타임은 GPU(T4)로 설정합니다.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# 2026-11-12 모델과 같은 버전 (metadata.yaml 의 version)
!pip install -q ultralytics==8.4.211


`data.yaml` 은 저장소의 `yolo/train/data.yaml` 을 Drive 의 `wildlife_yolo_project/dataset/` 에 올려 둔 것을 씁니다.


In [ ]:
from ultralytics import YOLO

DATA = '/content/drive/MyDrive/wildlife_yolo_project/dataset/data.yaml'
IMGSZ = 640
EPOCHS = 100   # 실제로 쓴 값으로 바꾸세요

model = YOLO('yolo26n.pt')
model.train(data=DATA, imgsz=IMGSZ, epochs=EPOCHS, project='/content/drive/MyDrive/wildlife_yolo_project/runs', name='animal_yolo26n')


In [ ]:
# 학습에 쓰지 않은 test 세트로 평가 (클래스별 recall 을 yolo/models/README.md 에 기록)
best = YOLO('/content/drive/MyDrive/wildlife_yolo_project/runs/animal_yolo26n/weights/best.pt')
metrics = best.val(data=DATA, imgsz=IMGSZ, split='test')
print(metrics.box.mr, metrics.box.map50)


In [ ]:
# Raspberry Pi 용 NCNN 으로 변환 → best_ncnn_model/ 폴더를 animal_yolo26n_ncnn 으로 이름 바꿔 Pi 에 복사
best.export(format='ncnn', imgsz=IMGSZ)
